# Crop Disease Detection - High Accuracy Model (Target: >95%)

This notebook implements an advanced Transfer Learning pipeline engineered to achieve **>95% accuracy** on the 17-class Crop Diseases dataset.

### Key Improvements over Baseline (`model.ipynb`):
1. **Pretrained Backbone (MobileNetV2)**: Uses ImageNet-learned representations instead of training shallow convolutional filters from scratch.
2. **Data Augmentation**: Incorporates random flips, rotations, zooms, and contrast variations to prevent overfitting.
3. **Two-Stage Training**:
   - **Phase 1 (Feature Extraction)**: Fast training of top classification layers with frozen backbone.
   - **Phase 2 (Fine-Tuning)**: Unfreezes top layers of the backbone with a micro learning rate (`1e-5`) for domain specialization.
4. **Adaptive Callbacks**: `ReduceLROnPlateau` to avoid overshooting optima, `EarlyStopping` with weight restoration, and `ModelCheckpoint` to save the best model (`.keras`).
5. **Complete Diagnostics**: Classification report (Precision, Recall, F1-Score), Confusion Matrix heatmap, and a Single-Image Inference helper.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
physical_devices = tf.config.list_physical_devices()
print("Available Physical Devices:", physical_devices)

## 1. Dataset Configuration & Loading
We use batch size 32 for better gradient stability, and (128, 128) image size for fast iteration.

In [ ]:
DATASET_DIR = "Crop Diseases"
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

# Training dataset (80% split)
training_set = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.2,
    subset="training",
    interpolation="bilinear"
)

# Validation dataset (20% split, un-shuffled for consistent evaluation)
validation_set = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False,
    seed=SEED,
    validation_split=0.2,
    subset="validation",
    interpolation="bilinear"
)

class_names = training_set.class_names
num_classes = len(class_names)
print(f"\nTotal Classes: {num_classes}")
print("Class List:", class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Performance caching and prefetching
train_ds = training_set.cache().shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)
val_ds = validation_set.cache().prefetch(buffer_size=AUTOTUNE)

## 2. Visualize Sample Images from the Training Set

In [ ]:
plt.figure(figsize=(12, 10))
for images, labels in training_set.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        class_idx = np.argmax(labels[i].numpy())
        plt.title(class_names[class_idx], fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Data Augmentation
Augmentation synthetically diversifies the dataset, preventing the model from memorizing exact pixel configurations.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
], name="data_augmentation")

# Preview data augmentation effects on a sample image
plt.figure(figsize=(10, 6))
for images, _ in training_set.take(1):
    sample_image = images[0]
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        augmented = data_augmentation(tf.expand_dims(sample_image, 0))
        plt.imshow(augmented[0].numpy().astype("uint8"))
        plt.title(f"Augmented Variation {i+1}")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Transfer Learning Architecture (MobileNetV2)
We attach a specialized classification head on top of the pre-trained MobileNetV2 base.

In [ ]:
input_shape = (IMAGE_SIZE[0], IMAGE_SIZE[1], 3)

# 1. Pretrained Base Model
base_model = MobileNetV2(
    input_shape=input_shape,
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze weights for Phase 1

# 2. Build Model Pipeline
inputs = layers.Input(shape=input_shape)
x = data_augmentation(inputs)
x = preprocess_input(x)  # Scales pixel values to [-1, 1] as required by MobileNetV2
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="CropDisease_MobileNetV2")
model.summary()

## 5. Phase 1: Feature Extraction (Base Frozen)
Train the new classification layers with Adam optimizer at learning rate `5e-4`.

In [ ]:
INITIAL_LR = 0.0005

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_list = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath="best_crop_disease_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

EPOCHS_PHASE_1 = 12

print("Starting Phase 1 Training...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_1,
    callbacks=callbacks_list
)

## 6. Phase 2: Fine-Tuning
Unfreeze the top layers of MobileNetV2 (from layer 100 onwards) and train at a reduced learning rate (`1e-5`). This refines high-level visual features for plant leaf diseases and pushes accuracy beyond 95%.

In [ ]:
base_model.trainable = True

# Fine-tune from layer 100 onwards
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total layers in base model: {len(base_model.layers)}")
print(f"Trainable layers count: {len(base_model.trainable_variables)}")

# Recompile with a very low learning rate
FINE_TUNE_LR = 1e-5
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_PHASE_2 = 10
total_epochs = len(history_phase1.history["accuracy"]) + EPOCHS_PHASE_2

print("Starting Phase 2 Fine-Tuning...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=len(history_phase1.history["accuracy"]),
    callbacks=callbacks_list
)

## 7. Training & Validation Performance Curves

In [ ]:
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']

loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#2ecc71', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#3498db', lw=2)
plt.axvline(x=len(history_phase1.history['accuracy']) - 1, color='red', linestyle='--', label='Fine-Tuning Start')
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Loss
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#e74c3c', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#f39c12', lw=2)
plt.axvline(x=len(history_phase1.history['loss']) - 1, color='red', linestyle='--', label='Fine-Tuning Start')
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Detailed Diagnostics: Classification Report & Confusion Matrix

In [ ]:
# Load best saved checkpoint
best_model = tf.keras.models.load_model("best_crop_disease_model.keras")
eval_loss, eval_acc = best_model.evaluate(val_ds, verbose=1)
print(f"\n>>> Best Model Validation Accuracy: {eval_acc * 100:.2f}% <<<")
print(f">>> Best Model Validation Loss: {eval_loss:.4f} <<<")

# Collect ground truth and model predictions
y_true = []
y_pred = []

for images, labels in validation_set:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Precision, Recall, F1-Score)")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Inference Function for New Images

In [ ]:
def predict_crop_disease(image_path, model, class_names, target_size=(128, 128)):
    """
    Predicts the disease for an input image path and displays the image with prediction.
    """
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)  # Add batch dimension

    predictions = model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    predicted_class = class_names[predicted_index]
    confidence = 100 * predictions[0][predicted_index]

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
    plt.axis("off")
    plt.show()

    return predicted_class, confidence

# Test on an example image from the dataset:
# sample_dir = os.path.join(DATASET_DIR, class_names[0])
# sample_file = os.path.join(sample_dir, os.listdir(sample_dir)[0])
# predict_crop_disease(sample_file, best_model, class_names)